# Librerías

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import re
from urllib.parse import urlsplit, urlunsplit
from tqdm import tqdm 
import sys 

tqdm.pandas()

In [3]:
sys.path.append('../')

In [4]:
from src.process_results.sorce_verification import url_exists
from src.process_results.source_identification import classify_link_peer_review, unknown_breakdown, refine_unknowns_df

# Constantes

In [5]:
PATH = '../data/'

# Lectura de datos

In [6]:
df_mistral = pd.read_csv(PATH + 'results_mistral.csv')

In [7]:
df_mistral.head()

,prompt,result,references,tokens
0,"I want to write an article about: ""Common fair...",Title: Incompatibility of Common Fairness Defi...,[],NaN
1,"I want to write an article about: ""Machine Lea...","Title: ""Unveiling Bias in Machine Learning: Th...",[],NaN
2,"I want to write an article about: ""Evaluation ...","Title: ""Fairness in Machine Learning: Beyond I...",[],NaN
3,"I want to write an article about: ""Benchmark c...","Title: ""Benchmarking Bias in AI: A Systematic ...",[],NaN
4,"I want to write an article about: ""Word embedd...",Title: Gender Bias in Word Embeddings: A Surve...,[],NaN


# Extract URLs

In [8]:
_URL_RE = re.compile(r'https?://[^\s<>"\']+')

def _clean_url(u: str) -> str:
    """
    Limpia basura común al final: ), ], }, ., ,, ;, :
    y normaliza levemente.
    """
    u = u.strip()

    # Recorta cierres típicos que se pegan en Markdown o puntuación final
    while u and u[-1] in ')]}.,;:':
        u = u[:-1]

    # Opcional: normaliza (sin tocar query/utm)
    # (esto evita cosas raras como espacios u otros, pero es suave)
    parts = urlsplit(u)
    return urlunsplit(parts)

def extract_urls_from_text(text) -> list[str]:
    """
    Devuelve una lista (orden de aparición) con todas las URLs encontradas en el texto.
    Maneja NaN/None.
    """
    if text is None:
        return []
    # Pandas puede traer NaN (float)
    if isinstance(text, float) and pd.isna(text):
        return []

    s = str(text)

    urls = []
    for m in _URL_RE.finditer(s):
        urls.append(_clean_url(m.group(0)))

    # De-duplicar manteniendo orden
    seen = set()
    out = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            out.append(u)
    return out

In [9]:
df_mistral["urls_clean"] = df_mistral["result"].apply(extract_urls_from_text)

In [10]:
df_mistral_urls = df_mistral.explode("urls_clean").dropna(subset='urls_clean')

# Existen los URL?

In [11]:
df_mistral_urls["url_check"] = df_mistral_urls["urls_clean"].progress_apply(url_exists)

100%|██████████| 475/475 [11:17<00:00,  1.43s/it]


In [14]:
def procesar_otras_status(url_check):
    url_check_copy = url_check.copy()
    if url_check['status_code'] in [200, 402, 403, 406, 502, 503, 405]:
        url_check_copy['exists'] = True
    return url_check_copy

df_mistral_urls['url_check'] = df_mistral_urls['url_check'].apply(procesar_otras_status)

In [15]:
df_mistral_urls["url_check"].apply(lambda x: x["exists"]).value_counts()

url_check
False    299
True     176
Name: count, dtype: int64

In [16]:
df_mistral_urls_exist = df_mistral_urls[df_mistral_urls["url_check"].apply(lambda x: x["exists"])]

In [17]:
df_mistral_urls_exist['url_check'].iloc[0]

{'url': 'https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing',
 'exists': True,
 'status_code': 200,
 'final_url': 'https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing',
 'error': None}

In [19]:
from src.process_results.classify_urls import classify_url                    

In [23]:
df_mistral_urls_revision = df_mistral_urls_exist[df_mistral_urls_exist.url_check.apply(lambda x: x['exists'])]
df_mistral_urls_revision['clasificacion'] = df_mistral_urls_revision['urls_clean'].progress_apply(
    classify_url)
print(df_mistral_urls_revision.shape)
# Nuevo
df_mistral_urls_revision['clasificacion'].apply(lambda x: x['peer_reviewed']).value_counts()

100%|██████████| 176/176 [00:02<00:00, 65.24it/s]

(176, 7)


clasificacion
no    135
sí     41
Name: count, dtype: int64

In [25]:
df_mistral_urls_revision[df_mistral_urls_revision['clasificacion'].apply(lambda x: x['peer_reviewed'])=='no'].urls_clean

17          https://psycnet.apa.org/record/1990-28631-001
17          https://mitpress.mit.edu/books/media-equation
17          https://psycnet.apa.org/record/2002-03648-001
67      https://www.theatlantic.com/magazine/archive/2...
115     https://medium.com/@paul_christiano/sycophanti...
                              ...                        
2018    https://www.veritas.com/content/dam/www/en_us/...
2023                            https://www.dans.knaw.nl/
2023                        https://www.thegreengrid.org/
2100             https://support.apple.com/en-us/HT209022
2121    https://www.americanbar.org/groups/law_schools...
Name: urls_clean, Length: 135, dtype: str

# Reference Verification (non-URL citations)

In [30]:
from src.process_results.reference_verification import (
    extract_references,
    extract_title,
    search_crossref,
    verify_reference,
    verify_references_df,
)

In [31]:
df_mistral

,prompt,result,references,tokens,urls_clean
0,"I want to write an article about: ""Common fair...",Title: Incompatibility of Common Fairness Defi...,[],NaN,[]
1,"I want to write an article about: ""Machine Lea...","Title: ""Unveiling Bias in Machine Learning: Th...",[],NaN,[https://www.propublica.org/article/machine-bi...
2,"I want to write an article about: ""Evaluation ...","Title: ""Fairness in Machine Learning: Beyond I...",[],NaN,[]
3,"I want to write an article about: ""Benchmark c...","Title: ""Benchmarking Bias in AI: A Systematic ...",[],NaN,[https://themarkup.org/artificial-intelligence...
4,"I want to write an article about: ""Word embedd...",Title: Gender Bias in Word Embeddings: A Surve...,[],NaN,[]
...,...,...,...,...,...
2130,"Define ""Positive bias"". Include references.","Positive bias, also known as optimism bias or ...",[],NaN,[]
2131,"Define ""Governance in AI"". Include references.","""Governance in AI"" refers to the systems, poli...",['[1] European Commission. (2018). Ethics Guid...,NaN,[https://ec.europa.eu/info/publications/ethics...
2132,"Define ""Intrinsic hallucinations"". Include ref...",Intrinsic hallucinations refer to hallucinatio...,"['[1] Frith, C. D., & Done, K. (2009). Halluci...",NaN,[]
2133,"Define ""Extrinsic hallucinations"". Include ref...",Extrinsic hallucinations refer to perceptions ...,['[1] American Psychiatric Association. (2013)...,NaN,[]


## Extract references per row

In [32]:
df_mistral["references_extracted"] = df_mistral["result"].apply(extract_references)

# Summary: how many citations found per row
ref_counts = df_mistral["references_extracted"].apply(len)
print(f"Total citations extracted: {ref_counts.sum()}")
print(f"Rows with ≥1 citation: {(ref_counts > 0).sum()} / {len(df_mistral)}")
print(f"Citations per row (mean): {ref_counts.mean():.1f}")
ref_counts.value_counts().sort_index()

Total citations extracted: 11499
Rows with ≥1 citation: 2135 / 2135
Citations per row (mean): 5.4


references_extracted
1      38
2     132
3     265
4     360
5     405
6     312
7     310
8     123
9     106
10     42
11     21
12      9
14      1
15      5
16      1
17      1
18      2
19      1
21      1
Name: count, dtype: int64

In [33]:
# Sample extracted references from first row with citations
first_with_refs = df_mistral[df_mistral["references_extracted"].apply(len) > 0].iloc[0]
print(f"Prompt: {first_with_refs['prompt'][:80]}...\n")
for i, ref in enumerate(first_with_refs["references_extracted"], 1):
    print(f"[{i}] {ref[:120]}")

Prompt: I want to write an article about: "Common fairness definitions are mathematicall...

[1] "Axiomatic Approaches to Fair Division" by Robert Aumann and Shmuel Rubinstein. Econometrica, vol. 50, no. 5, 1982, pp. 
[2] "Fair Division and the Cake Cutting Problem" by Robert Aumann and Moshe Tennenholtz. Games and Economic Behavior, vol. 1
[3] "Proportionality, Equality, and the Core" by Martin J. Osborne and Ariel Rubinstein. Econometrica, vol. 54, no. 3, 1986,
[4] "Envy-Freeness and Equality" by Leonid Hurwicz, Eric Maskin, and Robert Myerson. Econometrica, vol. 57, no. 5, 1989, pp.
[5] "Fair Division and the Shapley Value" by Martin J. Osborne and Ariel Rubinstein. Journal of Economic Theory, vol. 34, no
[6] "Fair Division and the Core" by Robert Aumann and Moshe Tennenholtz. Journal of Economic Theory, vol. 38, no. 1, 1987, p
[7] "Fair Division and the Core: A Survey" by Robert Aumann and Moshe Tennenholtz. Journal of Economic Surveys, vol. 1, no. 


## Run reference verification (CrossRef lookup)

In [34]:
SAVE_PATH = PATH + "processed/mistral_reference_checks.csv"

df_ref_checks = verify_references_df(
    df_mistral,
    save_path=SAVE_PATH,
    save_every=50,
)
df_ref_checks.head()

  Checkpoint saved at row 50/11499
  Checkpoint saved at row 100/11499


KeyboardInterrupt: 

## Summary statistics

In [ ]:
status_counts = df_ref_checks["status"].value_counts()
status_pct = (status_counts / len(df_ref_checks) * 100).round(1)
summary = pd.DataFrame({"count": status_counts, "pct": status_pct})
print(f"Total citations verified: {len(df_ref_checks)}\n")
print(summary)

Total citations verified: 72

              count   pct
status                   
review           48  66.7
hallucinated     20  27.8
exists            4   5.6


## Examples by category

In [ ]:
def show_examples(status, n=3):
    subset = df_ref_checks[df_ref_checks["status"] == status].head(n)
    print(f"=== {status.upper()} (n={len(df_ref_checks[df_ref_checks['status']==status])}) ===\n")
    for _, row in subset.iterrows():
        print(f"Citation       : {row['citation'][:100]}")
        print(f"Title (Mistral): {row['title_extracted']}")
        print(f"Venue (Mistral): {row['venue_extracted'] or '(not found)'}")
        print(f"Title sim      : {row['title_similarity']}")
        print(f"Venue (CrossRef): {row['venue_crossref'] or '(not found)'}")
        print(f"Venue sim      : {row['venue_similarity']}")
        if row["top_match"]:
            m = row["top_match"]
            print(f"CrossRef title : {m.get('title', '')[:80]} ({m.get('published_year')}) [{m.get('type')}]")
        print()

for s in ["exists", "review", "hallucinated"]:
    print('>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>', s)
    if s in df_ref_checks["status"].values:
        show_examples(s)

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> exists
=== EXISTS (n=4) ===

Citation       : Barocas, S., & Selbst, A. (2016). Big data's hidden biases. Communications of the ACM, 59(10), 80-87
Title (Mistral): Fairness in Machine Learning: A Survey
Venue (Mistral): Communications of the ACM
Title sim      : 1.0
Venue (CrossRef): ACM Computing Surveys
Venue sim      : 0.3913
CrossRef title : Fairness in Machine Learning: A Survey (2024) [journal-article]

Citation       : Chouldechova, A. (2017). The limits of fair machine learning. Proceedings of the National Academy of
Title (Mistral): Fairness in Algorithmic Decision Making: A Survey
Venue (Mistral): Proceedings of the National Academy of Sciences
Title sim      : 0.8864
Venue (CrossRef): Proceedings of the 7th ACM IKDD CoDS and 25th COMAD
Venue sim      : 0.5102
CrossRef title : Fairness in Algorithmic Decision Making (2020) [proceedings-article]

Citation       : Caliskan, Aylin, et al., 2017. "Semantics derived automatically from language 

## Re-run: only "review" and "exists" with fixed title extraction

In [ ]:
from src.process_results.reference_verification import extract_title, extract_venue, verify_reference

# Load existing results
df_checks = pd.read_csv(PATH + "processed/mistral_reference_checks.csv")
print(f"Total rows loaded: {len(df_checks)}")
print(df_checks["status"].value_counts())

Total rows loaded: 11499
status
hallucinated    5314
review          5271
exists           914
Name: count, dtype: int64


In [ ]:
mask = df_checks["status"].isin(["review", "hallucinated"])
to_rerun = df_checks[mask].copy()
print(f"Re-running {len(to_rerun)} citations ({mask.sum()/len(df_checks)*100:.1f}% of total)")

Re-running 10585 citations (92.1% of total)


In [ ]:
SAVE_PATH_V2 = PATH + "processed/mistral_reference_checks_V2.csv"

rerun_results = []
for i, (idx, row) in enumerate(to_rerun.iterrows()):
    result = verify_reference(row["citation"])
    rerun_results.append({
        "original_index": row["original_index"],
        "prompt": row["prompt"],
        **result,
    })
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(to_rerun)}")

df_rerun = pd.DataFrame(rerun_results)
print(f"\nDone. New status distribution:")
print(df_rerun["status"].value_counts())

  50/10585
  100/10585
  150/10585
  200/10585
  250/10585
  300/10585
  350/10585
  400/10585
  450/10585
  500/10585
  550/10585
  600/10585
  650/10585
  700/10585
  750/10585
  800/10585
  850/10585
  900/10585
  950/10585
  1000/10585
  1050/10585
  1100/10585
  1150/10585
  1200/10585
  1250/10585
  1300/10585
  1350/10585
  1400/10585
  1450/10585
  1500/10585
  1550/10585
  1600/10585
  1650/10585
  1700/10585
  1750/10585
  1800/10585
  1850/10585
  1900/10585
  1950/10585
  2000/10585
  2050/10585
  2100/10585
  2150/10585
  2200/10585
  2250/10585
  2300/10585
  2350/10585
  2400/10585
  2450/10585
  2500/10585
  2550/10585
  2600/10585
  2650/10585
  2700/10585
  2750/10585
  2800/10585
  2850/10585
  2900/10585
  2950/10585
  3000/10585
  3050/10585
  3100/10585
  3150/10585
  3200/10585
  3250/10585
  3300/10585
  3350/10585
  3400/10585
  3450/10585
  3500/10585
  3550/10585
  3600/10585
  3650/10585
  3700/10585
  3750/10585
  3800/10585
  3850/10585
  3900/10585
  3950

In [63]:
# Merge: keep "exists" as-is, replace review/hallucinated with re-run results
df_exists = df_checks[~mask].copy()
df_final = pd.concat([df_exists, df_rerun], ignore_index=True)

df_final.to_csv(SAVE_PATH_V2, index=False)
print(f"Saved {len(df_final)} rows to {SAVE_PATH_V2}")
print("\nFinal status distribution:")
print(df_final["status"].value_counts(True))

Saved 11499 rows to data/processed/mistral_reference_checks_V2.csv

Final status distribution:
status
review          0.485781
hallucinated    0.269850
exists          0.244369
Name: proportion, dtype: float64


In [62]:
for s in ["exists", "review", "hallucinated"]:
    print('>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>', s)
    if s in df_ref_checks["status"].values:
        show_examples(s)

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> exists
=== EXISTS (n=914) ===

Citation       : Barocas, S., & Selbst, A. (2016). Big data's hidden biases. Communications of the ACM, 59(10), 80-87
Title (Mistral): Fairness in Machine Learning: A Survey
Venue (Mistral): Communications of the ACM
Title sim      : 1.0
Venue (CrossRef): ACM Computing Surveys
Venue sim      : 0.3913
CrossRef title : Fairness in Machine Learning: A Survey (2024) [journal-article]

Citation       : Chouldechova, A. (2017). The limits of fair machine learning. Proceedings of the National Academy of
Title (Mistral): Fairness in Algorithmic Decision Making: A Survey
Venue (Mistral): Proceedings of the National Academy of Sciences
Title sim      : 0.8864
Venue (CrossRef): Proceedings of the 7th ACM IKDD CoDS and 25th COMAD
Venue sim      : 0.5102
CrossRef title : Fairness in Algorithmic Decision Making (2020) [proceedings-article]

Citation       : Caliskan, Aylin, et al., 2017. "Semantics derived automatically from languag